# abcGPT-nano-3 — the 100-slider "slider" language model

This is a **GPT-2-scale (124M-base, ~443M total) language model** with **100 per-domain LoRA "sliders"** bolted on.
Each slider corresponds to one of 100 topic/domain clusters mined from FineWeb. At generation time you hand the
model a length-100 weight vector **alpha** (over the simplex). The model adds
`sum_c alpha[c] * cap[c] * (U_c @ V_c^T) * rslora_scale` into every gated linear, so you can **steer the topic and
style of the text** by deciding which clusters to turn on and how hard.

- One-hot alpha (all mass on cluster *k*) -> steer fully toward cluster *k*.
- Spread the mass over several clusters -> **blend** their styles. Arbitrary mixes work.

### Mid-training caveat (read this)
The checkpoint here (`slider-gpt2-bRfull-commit100`) is **mid-training: iter ~180k of 610k**. The base LM is
coherent-ish for its size, but the per-slider deltas are still small and growing. **Steering is real but subtle** —
expect nudges in vocabulary/register, not dramatic genre switches. Crank the strength and use the side-by-side
compare tool to actually see it. This is a research demo, not a polished product.

Runtime: pick a **GPU runtime** in Colab (Runtime -> Change runtime type -> T4 GPU). CPU works but is slow.

In [ ]:
!pip install -q torch tiktoken huggingface_hub ipywidgets numpy

In [ ]:
import os, json, urllib.request
from huggingface_hub import hf_hub_download

REPO = "iamtrask/abcGPT-nano-3"
CKPT = "slider-gpt2-bRfull-commit100/ckpt.pt"
LABELS = "fineweb_cluster/cluster_labels.json"
MODEL_PY_URL = "https://raw.githubusercontent.com/iamtrask/abcGPT/fineweb-cluster/experiments/nano-3/model.py"

# HF_TOKEN: the repo may be public, so try without a token first and fall back
# to the env token if the anonymous download fails.
HF_TOKEN = os.environ.get("HF_TOKEN")  # set this in Colab Secrets if the repo is private

# 1) model.py (defines NanoGPT / NanoGPTConfig)
urllib.request.urlretrieve(MODEL_PY_URL, "model.py")
print("downloaded model.py")

def _download(repo, filename, token):
    try:
        return hf_hub_download(repo, filename)  # anonymous
    except Exception as e:
        if not token:
            raise
        print(f"  anon download of {filename} failed ({type(e).__name__}); retrying with token")
        return hf_hub_download(repo, filename, token=token)

# 2) checkpoint (~1.77 GB) and 3) cluster labels
ckpt_path = _download(REPO, CKPT, HF_TOKEN)
labels_path = _download(REPO, LABELS, HF_TOKEN)
print("checkpoint:", ckpt_path)
print("labels:", labels_path)

# cluster_labels.json is a dict keyed by string indices "0".."99". Each value is
# {"n_domains": int, "tokens": int, "top_domains": [str, ...]}. There is no single
# human label, so we build a readable label from the top domains of each cluster.
raw_labels = json.load(open(labels_path))

def _short(domain):
    # strip leading www. and the TLD for a compact tag, e.g. articles.latimes.com -> latimes
    d = domain.lower()
    for pre in ("www.", "www2.", "articles.", "blog.", "transcripts."):
        if d.startswith(pre):
            d = d[len(pre):]
    parts = d.split(".")
    # take the registrable-ish middle token
    return parts[-2] if len(parts) >= 2 else parts[0]

def cluster_label(i):
    info = raw_labels[str(i)]
    tops = info.get("top_domains", [])[:3]
    tags = ", ".join(dict.fromkeys(_short(t) for t in tops))  # dedupe, keep order
    return f"cluster_{i:02d}: {tags}" if tags else f"cluster_{i:02d}"

LABELS_LIST = [cluster_label(i) for i in range(100)]
print("\nexample labels:")
for i in (0, 1, 2, 50, 99):
    print(" ", LABELS_LIST[i])

In [ ]:
import torch
from model import NanoGPT, NanoGPTConfig

# This config matches exactly what the checkpoint was trained with. CRITICAL:
# offload_deltas=True so the per-cohort deltas are stored as ParameterLists
# (keys U.0, U.1, ... / V.0, V.1, ...) that match the checkpoint state_dict.
cfg = NanoGPTConfig(
    n_layer=12, n_head=12, n_embd=768, block_size=1024, vocab_size=50304,
    variant="lora", n_cohorts=100,
    cohort_names=[f"cluster_{i:02d}" for i in range(100)],
    rank=16, base_rank=-1, adaptive_capacity=True, bias_anchor=True,
    rslora=True, gate_attention=True, gate_embedding=True,
    offload_deltas=True, reserve_frac=0.0,
)

model = NanoGPT(cfg)

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
sd = ckpt["model"]
# strip a torch.compile prefix if present (this checkpoint has none, but be safe)
sd = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in sd.items()}

# strict=True is expected to pass cleanly for this checkpoint+config. If it ever
# reports only a couple of missing *buffers* (e.g. base_gate), fall back to
# strict=False -- but any missing/unexpected *parameter* means the config is wrong.
try:
    model.load_state_dict(sd, strict=True)
    print("load_state_dict: strict=True OK (exact match)")
except RuntimeError as e:
    print("strict load failed, retrying strict=False:", str(e)[:300])
    res = model.load_state_dict(sd, strict=False)
    print("  missing:", res.missing_keys)
    print("  unexpected:", res.unexpected_keys)

model.eval()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nmodel loaded on {DEVICE} | params: {n_params:,} | training iter: {ckpt.get('iter')}")
print("(mid-training checkpoint -- steering will be real but subtle)")

In [ ]:
import tiktoken
import torch.nn.functional as F
import numpy as np

enc = tiktoken.get_encoding("gpt2")  # model vocab is padded to 50304; real tokens < 50257

@torch.no_grad()
def generate(prompt, alpha, max_new_tokens=80, temperature=0.8, top_k=40):
    """Autoregressively sample from the model under a fixed length-100 alpha.

    alpha: list/np.array/tensor of length 100 (n_cohorts). It is used as-is every
    step. Mass on cluster k steers toward that cluster's domain/style; spread the
    mass to blend. Does NOT need to sum to 1, but staying near the simplex keeps
    the delta magnitude in the trained regime.
    """
    a = torch.as_tensor(alpha, dtype=torch.float32, device=DEVICE)
    ids = enc.encode(prompt)
    idx = torch.tensor(ids, dtype=torch.long, device=DEVICE)[None, :]
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -cfg.block_size:]  # crop context to block_size
        logits, _ = model(idx_cond, alpha=a)
        logits = logits[:, -1, :] / max(temperature, 1e-5)
        if top_k is not None and top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, nxt], dim=1)
    return enc.decode(idx[0].tolist())


def make_alpha(primary, strength=1.0, secondary=None, blend=0.0):
    """Build a length-100 alpha vector.

    primary:   cluster index to put most mass on.
    strength:  how much of the budget goes to the primary (0..1). The remainder
               goes to the secondary if given, else is spread uniformly over all
               clusters (a mild 'average style' floor).
    secondary: optional second cluster index to blend in.
    blend:     fraction of the *remainder* that goes to the secondary (0..1).
    """
    a = np.zeros(100, dtype=np.float32)
    strength = float(np.clip(strength, 0.0, 1.0))
    a[primary] = strength
    rem = 1.0 - strength
    if secondary is not None and secondary != primary and rem > 0:
        a[secondary] += rem * float(np.clip(blend, 0.0, 1.0))
        rem_after = rem * (1.0 - float(np.clip(blend, 0.0, 1.0)))
    else:
        rem_after = rem
    if rem_after > 0:
        a += rem_after / 100.0  # spread the leftover uniformly
    return a

# quick smoke test
_demo = make_alpha(0, strength=1.0)
print("smoke test (one-hot cluster 0):")
print(generate("The latest news today is", _demo, max_new_tokens=40))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Dropdown options: (label, index)
_options = [(LABELS_LIST[i], i) for i in range(100)]

primary_dd   = widgets.Dropdown(options=_options, value=0, description="Primary:",
                                layout=widgets.Layout(width="520px"))
strength_sl  = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05,
                                   description="Strength:", continuous_update=False,
                                   layout=widgets.Layout(width="520px"))
secondary_dd = widgets.Dropdown(options=[("(none)", None)] + _options, value=None,
                                description="Secondary:", layout=widgets.Layout(width="520px"))
blend_sl     = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                                   description="Blend:", continuous_update=False,
                                   layout=widgets.Layout(width="520px"))

prompt_ta    = widgets.Textarea(value="The latest news today is", description="Prompt:",
                                layout=widgets.Layout(width="640px", height="70px"))
temp_sl      = widgets.FloatSlider(value=0.8, min=0.1, max=1.5, step=0.05,
                                   description="Temp:", continuous_update=False,
                                   layout=widgets.Layout(width="520px"))
tokens_sl    = widgets.IntSlider(value=80, min=10, max=300, step=10,
                                 description="Tokens:", continuous_update=False,
                                 layout=widgets.Layout(width="520px"))
go_btn       = widgets.Button(description="Generate", button_style="primary")
out          = widgets.Output()

def _current_alpha():
    return make_alpha(primary_dd.value, strength_sl.value,
                      secondary_dd.value, blend_sl.value)

def _on_go(_):
    with out:
        clear_output()
        a = _current_alpha()
        active = [(i, round(float(a[i]), 3)) for i in range(100) if a[i] > 0.01]
        print("active sliders (idx, weight):", active)
        print("-" * 70)
        txt = generate(prompt_ta.value, a, max_new_tokens=tokens_sl.value,
                       temperature=temp_sl.value)
        print(txt)

go_btn.on_click(_on_go)

display(widgets.VBox([
    widgets.HTML("<b>Topic sliders</b> -- choose a primary cluster and how hard to push it; "
                 "optionally blend a secondary."),
    primary_dd, strength_sl, secondary_dd, blend_sl,
    widgets.HTML("<b>Generation</b>"),
    prompt_ta, temp_sl, tokens_sl, go_btn, out,
]))

## Side-by-side compare

Steering on a mid-training checkpoint is subtle. The easiest way to *see* it is to hold the prompt fixed and
generate under several different slider settings with the **same random seed**, so any difference is the alpha, not
sampling noise. Edit `SETTINGS` below and run.

In [ ]:
def compare(prompt, settings, max_new_tokens=80, temperature=0.8, seed=0):
    """settings: list of (title, alpha_vector). Generates each under the same seed."""
    for title, a in settings:
        torch.manual_seed(seed)
        print("=" * 78)
        print(title)
        print("=" * 78)
        print(generate(prompt, a, max_new_tokens=max_new_tokens, temperature=temperature))
        print()

PROMPT = "The latest news today is"

# Try three different single-cluster steers + a blend. Change the indices to any 0..99;
# print LABELS_LIST[i] to see what each cluster is about.
SETTINGS = [
    (f"PRIMARY {LABELS_LIST[2]}",  make_alpha(2,  strength=1.0)),
    (f"PRIMARY {LABELS_LIST[50]}", make_alpha(50, strength=1.0)),
    (f"PRIMARY {LABELS_LIST[99]}", make_alpha(99, strength=1.0)),
    (f"BLEND 2 + 50",              make_alpha(2, strength=0.5, secondary=50, blend=1.0)),
]

compare(PROMPT, SETTINGS, max_new_tokens=80)

## How to read the results

- **alpha is the whole point.** Each generation is the *same* base network with a different length-100 weight
  vector added into its linear layers. If two outputs differ under a fixed seed and prompt, that difference *is* the
  steering.
- **One-hot vs blend.** Put all the mass on one cluster for the strongest (still subtle) steer; spread it to blend
  registers. `make_alpha(primary, strength, secondary, blend)` builds the vector; you can also hand `generate` any
  100-length array you construct yourself.
- **Subtlety is expected.** This checkpoint is ~180k/610k iters in; the per-cluster deltas are still small. Look for
  shifts in vocabulary, named entities, and register (newswire vs forum vs reference) rather than hard genre flips.
- **Clusters are FineWeb domain clusters.** The label for each is derived from its top source domains
  (`LABELS_LIST[i]`), so e.g. a cluster dominated by newspaper sites will tug the text toward news prose.